# InnerSource StateCalc Example <a name="top"></a>

## Table of Contents
1. [Introduction](#introduction)
2. [End-to-end StateCalc Example and Explanations](#example)
    - [Libraries and helper functions](#libraries)
    - [Local paths](#paths)
    - [2.1 Prepare input events](#inputevents)
    - [2.2 Define statecalc foundation attributes](#statecalc-foundation-attributes)
    - [2.3 Define statecalc calculation attributes](#statecalc-calculation-attributes)
    - [2.4 Design Aggregation function](#statecalc-aggregate)
    - [2.5 Run statecalc](#runstatecalc)
    - [2.6 Review statecalc outputs](#output)
3. [Conclusion](#conclusion)

## Introduction <a name="introduction"></a>
State Calculation creates a ”Behavioral Profile” for an entity after events have been synthesized and checked for consistency. It enriches, transforms and aggregates synthesized data. The State calculation stage provides a wide range of options to configure the the states using configurations and for advanced state sections/fields custom code can be written to compute data. State calculation is performed in 2 passes. Foundation Stage and Calculation stage.
EXperian Profile Map (XPM) – Output of the State Calculation which provides a view served by API for viewing the State Model.

<img src="./example/images/Statecalc_flow.png" alt="StateCalculation Flowchart" width="600"/>

[Return to Top](#top)

## End-to-end StateCalc Example and Explanations <a name="example"></a>
The StateCalc job orchestrates through the following steps:

1. [**Prepare input events**](#inputevents)
2. [**Define foundation attributes**](#statecalc-foundation-attributes)
3. [**Define state calculation attributes**](#statecalc-calculation-attributes)
4. [**Define Aggregate function**](#statecalc-aggregate)
5. [**Run StateCalculation**](#runstatecalc)
6. [**Review statecalc outputs**](#output)


#### Libraries and helper functions <a name="libraries"></a>
First, we begin by importing necessary Python libraries and helper functions. These aid in extracting key information from inputs, intermediary results, and final outputs.

In [1]:
import os
import re
import json
import pandas as pd
pd.set_option('display.max_colwidth', None)

def print_jsonl_1stline(file):
    with open(file, 'r') as f:
        line = f.readline()
        data = json.loads(line)
        print(json.dumps(data, indent=2))

def retrieve_statecalc_attributes(statecalc_foundation_path):
    f =  open(statecalc_foundation_path, 'r')
    data = json.load(f)
    print(json.dumps(data, indent=2))
    f.close()


def retrieve_json_fields_statecalc(output_list, array_element_field1, array_element_field2, attributes):
    df = pd.read_json(output_list, lines=True)
    df1 = df[[c for c in df.columns if c in attributes]]
    return df1

def filter_files(files, substring):
    for file in files:
        if substring in file:
            return file

def print_jsonl_by_pin(file, pin):
    with open(file, 'r') as f:
        for line in f:
            data = json.loads(line)
            if data.get("pin") == pin:
                print(json.dumps(data, indent=2))
                break

[Return to Top](#top)

#### Local paths <a name="paths"></a>
Subsequently, please revise the path variables to ensure they correctly point to the respective paths of the input, output, and identifying attributes configuration file on your machine.

In [2]:
input_folder = 'example/syntehtic-events-input/consistency/synthesized/'
output_root_path = 'example/output-demo'
statecalc_foundation_path = 'src/main/resources/statecalc-foundation_sample.json'
statecalc_calculation_path = 'src/main/resources/statecalc-calculation_sample.json'

[Return to Top](#top)

### 1. Prepare input events <a name="inputevents"></a>
The inputs for the statecalc process are synthetic events - events that have been synthesize after pinning. 

Each event is assigned a unique "eventId", with all event details stored as values in various attribute fields.

We read from the inputs and display the first event as an example.

In [3]:
# Print the first input event as an example
files = [file for file in os.listdir(input_folder) if file.endswith('.json')]
file = os.path.join(input_folder, files[0])
print_jsonl_1stline(file)

{
  "pin": 2002,
  "synthesizedEvents": [
    {
      "asOfPeriod": {
        "epochDateTime": 1485993600000,
        "id": "201702",
        "unit": {
          "name": "MonthOfYear"
        }
      },
      "curatedRecordJson": {
        "ageOfMora": 360,
        "balance": 26530,
        "creditLimit": 0,
        "dueDate": 20301231,
        "firstEventDate": 1485993600000,
        "firstLastName": "VANEGAS",
        "firstName": "ALBEIRO",
        "isNewSubscriberRelation": false,
        "middleName": "",
        "nationalId": "00006235083",
        "paymentDate": 20080718,
        "refersToRelation": "",
        "secondLastName": "GONZALEZ"
      },
      "dataSource": "Oxygen",
      "frequency": {
        "count": 1,
        "unit": {
          "name": "EveryNMonth"
        }
      },
      "migrationHistory": [],
      "parentEventIds": [
        "31555bbf-e851-af01-0000-000000000008"
      ],
      "subscriber": "230001",
      "subscriberRelation": "000000111114092120"
    }

[Return to Top](#top)

### 2. Define statecalc foundation attributes <a name="statecalc-foundation-attributes"></a>
"Foundation State" is used to transform / enrich existing synthetics events. It takes synthesized events as input and produces a sparse metrics of all the fields required for the next phase(calculation).

There is one synthetic event per datasource - subscriber - subscriber relation - asOfPeriod (currently defaulting to month) coming from consistency check. This means that for a given datasource, there will be one event for Citibank (Subscriber) - Account number 11113333 (subscriber relation) for Feb 2018 (asOfPeriod)

Below, we provide an example of the statecalc-foundation attributes in use.

In [4]:
# Print StateCacl Foundation attributes
print("StateCalc Foundation Attributes:")
statecalc_foundation_attributes = retrieve_statecalc_attributes(statecalc_foundation_path)

StateCalc Foundation Attributes:
{
  "domain": "creaditbureau",
  "fields": [
    {
      "name": "subscriber",
      "description": "Subscriber code",
      "aliases": [
        "subscriberCode",
        "counterpartyCode"
      ]
    },
    {
      "name": "cutoffDateDouble",
      "description": "The cutoff date Schema",
      "dataType": "DOUBLE",
      "type": "CALCULATED",
      "value": "return field.cutoffDate"
    },
    {
      "name": "isMasterFile",
      "description": "Check for Master files",
      "type": "CALCULATED",
      "defaultValue": "false",
      "value": "return (field.isCredit || field.isExpress || field.isCheckingAndSaving || field.isCustomersMaster || field.isContactMaster)",
      "dataType": "BOOLEAN"
    },
    {
      "name": "balance",
      "description": "balance",
      "defaultValue": "1000.00",
      "dataType": "DOUBLE"
    }
  ]
}


Calculated Field
For calculated types, the user is expected to provide the name of the new field to be added to each synthetic event and the value field is tied to a Nashhorn Javascript Engine which evaluates and populates the new field value. Here is a snippet of code to describe this configuration. This configuration will produce another column called "calculatedBalance" in each synthetic event and its value will be 0.9 * balance. The "value" field will supports JavaScript expressions used for transformations / calculations which is demonstrated in the second transformation for the 'businessLineCode' field. 

PS: 'field' is a variable containing all fields of current Event

In [7]:
#Define "complex_field.nested_field"  to access nested field in Statecalc-foundation.json  After that, used the "nested_field" directly as type : CALCULATED .
{
     "name": "officeOfFiling.mainCity",
     "description": "office of filling city "
},
{
     "name": "officeOfFiling.number",
     "description": "office of filling number "
},
{
     "name": "officeMainCity",
     "description": "main city from complex nested field",
     "type": "CALCULATED",
     "value": "if(field.mainCity) { return field.mainCity }"
},
{
     "name": "square_office_number",
     "fieldType": "Long",
     "fieldType": "CALCULATED",
     "value": "return field.number * field.number"
}

{'name': 'square_office_number',
 'fieldType': 'CALCULATED',
 'value': 'return field.number * field.number'}

In [ ]:
# Nashorn Construct Javacript Function
def constructCalculatedFieldJsFunction(engine: ScriptEngine, foundationField: FoundationField): Unit = {
    val value =
      s"""function foundation_calculated_${foundationField.name}(field){
         |  ${foundationField.value}
         |}""".stripMargin

    val compiledScript: CompiledScript = engine.asInstanceOf[Compilable].compile(value)
    compiledScript.eval()
    logger.debug(s"Registered Javascript Function to Engine for Foundation Field ${foundationField.name()}(CALCULATED):\n $value")
  }
  

In [ ]:
# Nashorn Compute Javascript Function
def computeCalculatedField(invokableEngine: Invocable, foundationField: FoundationField, foundationStateModel: JavaFoundationRecord): Option[Any] = {
    val value = try
      invokableEngine.invokeFunction("foundation_calculated_" + foundationField.name, foundationStateModel)
    catch {
      case e: Exception => throw new StateCalculationScriptExecutionException(e.getMessage + " - In Field : " + foundationField.name + " of foundation config")
    }
    Option(toDataType(foundationField.dataType(), value))
  }


### 3. Define statecalc calculation attributes <a name="statecalc-calculation-attributes"></a>
After foundation state is computed it will generate a sparse matrix style data, the calculated stage is based on foundation data. 
Calculation config is a JSON file where different aggregation calculated sections can be defined using configuration. Configuration sections is modeled in a way it invokes SQL like principles. Json Schema for defining Aggregation config is as follows.


In [5]:
# Print StateCacl Calculation attributes
print("StateCalc Calculation Attributes:")
statecalc_calculation_attributes = retrieve_statecalc_attributes(statecalc_calculation_path)

StateCalc Calculation Attributes:
{
  "domain": "credit",
  "calculations": [
    {
      "name": "AggregationOnBalanceGroupBySubSR",
      "description": "Check Information",
      "type": "AGGREGATES",
      "select": [
        "subscriber",
        "subscriberRelation"
      ],
      "groupBy": [
        "subscriber",
        "subscriberRelation"
      ],
      "aggregates": [
        {
          "function": "SUM",
          "field": "balance",
          "name": "sum"
        },
        {
          "function": "AVERAGE",
          "field": "balance",
          "name": "avg"
        },
        {
          "function": "MIN",
          "field": "balance",
          "name": "min"
        }
      ]
    },
    {
      "name": "names",
      "description": "Names of an individual",
      "type": "MOST_RECENT_VALUE",
      "select": [
        "counterpartyIdType",
        "counterpartyIdNumber",
        "personIdType",
        "personIdNumber",
        "businessLineCode",
        "dataSourc

[Return to Top](#top)

### 3. Design StateCalc Calculation and Aggregation Function <a name="statecalc-aggregate"></a>
This aggregation type provides you an option to to compute aggregation functions like MIN/MAX/SUM/AVERAGE/COUNT on the data.

Generates :

Array, if groupBy clause is mentioned
Object, if no Groupby clause is mentioned.
Notes:

If GroupBy clause is defined, Only fields provided in groupby clause are allowed in Select 
If groupBy clause is not defined, Select clause is not allowed.

<img src="./example/images/statecalc_agg.png" alt="StateCalc Agg" width="900"/>

In [6]:
#Print the sample foundation input to statecalc
foundation_path = 'example/statecalc/sample.json'
 
array_element_field1 = 'subscriber'
array_element_field2 = 'subscriberRelation'
attributes = ['pin','firstName','secondLastName','balance','subscriber','subscriberRelation']
df = retrieve_json_fields_statecalc(foundation_path, array_element_field1, array_element_field2, attributes)
print(df)

   balance firstName   pin secondLastName  subscriber  subscriberRelation
0    26530   ALBEIRO  2002       GONZALEZ      230001        111114092120
1    26530   ALBEIRO  2002       GONZALEZ      230001        111114092121
2    26530   ALBEIRO  2002       GONZALEZ      230001        111114092120


In [7]:
# Print state calc computation output
foundation_path = 'example/statecalc/sample.json'
 
array_element_field1 = 'subscriber'
array_element_field2 = 'subscriberRelation'
attributes = ['pin','firstName','secondLastName','balance','subscriber','subscriberRelation']
 
df = retrieve_json_fields_statecalc(foundation_path, array_element_field1, array_element_field2, attributes)

for g, data in df.groupby(['subscriber','subscriberRelation']):
    print(g, data)


(230001, 111114092120)    balance firstName   pin secondLastName  subscriber  subscriberRelation
0    26530   ALBEIRO  2002       GONZALEZ      230001        111114092120
2    26530   ALBEIRO  2002       GONZALEZ      230001        111114092120
(230001, 111114092121)    balance firstName   pin secondLastName  subscriber  subscriberRelation
1    26530   ALBEIRO  2002       GONZALEZ      230001        111114092121


In [8]:
# Print state calc xpm aggregation output
foundation_path = 'example/statecalc/agg.json'
df = pd.read_json(foundation_path, lines=True)
agg = df['AggregationOnBalanceGroupBySubSR']
agg.head(1)


0    [{'avg': 26530.0, 'min': 26530, 'subscriber': '230001', 'subscriberRelation': '000000111114092120', 'sum': 26530}, {'avg': 26530.0, 'min': 26530, 'subscriber': '230001', 'subscriberRelation': '000000111114092121', 'sum': 26530}]
Name: AggregationOnBalanceGroupBySubSR, dtype: object

[Return to Top](#top)

### 4. Run Statecalc <a name="runstatecalc"></a>
Run statecalc with sample input, foundation configurations, statecalc configurations and Agg function

In [12]:
# Uncomment the lines in this block and run statecalc
# !cp /data/src/main/resources/log4j.properties /opt/spark/conf/;
# !spark-submit --master local[2] --driver-memory 2g --executor-memory 2g --class com.experian.edf.oxygen.statecalculation.app.StateCalcApp target/oxygen-statecalc-innersource-1.0-jar-with-dependencies.jar env.conf statecalc-application.conf;

[Return to Top](#top)

### 5. Review statecalc outputs <a name="output"></a>
The final output of statecalc witll calculate all functions mentioned in statecalculation configuraton.
 

In [9]:
# Print one of the pins in the output
output_root_path = 'example/statecalc/output.json'
f =  open(output_root_path, 'r')
data = json.load(f)
print(json.dumps(data, indent=2))
f.close()

{
  "AggregationOnBalanceGroupBySubSR": [
    {
      "avg": 26530.0,
      "min": 26530,
      "subscriber": "230001",
      "subscriberRelation": "000000111114092120",
      "sum": 26530
    },
    {
      "avg": 26530.0,
      "min": 26530,
      "subscriber": "230001",
      "subscriberRelation": "000000111114092121",
      "sum": 26530
    }
  ],
  "parentEventIds": [
    "31555bbf-e851-af01-0000-000000000008"
  ],
  "pin": 2002
}


[Return to Top](#top)

## Conclusion <a name="conclusion"></a>
State Calculation will enrich the synthetic events and calculate the xpm based on functions mentioned in statecalclation configurations.


[Return to Top](#top)